# Training Comparison

**Experiment Note:** 

- **In this notebook, you have to restart kernel before training each model to clear memory, remove cache, and avoid OOM errors.**
- **Must Run Configurations Every times Before Training The Model**

### Training Configurations

In [ ]:
# Training Comparison Notebook Configuration
import sys
import os
import json
sys.path.append("..")

# Dataset parameters (used for preprocessing)
MAX_SEQ_LEN              = 384
MINORITY_CONFIDENCE      = 0.99
MINORITY_MARGIN_ERROR    = 0.0130

# Training hyperparameters
PER_DEVICE_BATCH         = 8
GRAD_ACCUM               = 8
LEARNING_RATE            = 1e-3
NUM_EPOCHS               = 2
WARMUP_STEPS             = 100
WEIGHT_DECAY             = 0.1
EARLY_STOPPING_PATIENCE  = 3
EARLY_STOPPING_THRESHOLD = 0.001

# Optimizer (AdamW)
ADAM_BETA1               = 0.9
ADAM_BETA2               = 0.95
ADAM_EPS                 = 1e-8
MAX_GRAD_NORM            = 1.0

# LR schedule
# From DeepSeekMoE paper (Training approach) -  the learning rate is multiplied by 0.316 at 80 % of the training steps, and again by 0.316 at 90 % of the training steps.
SCHEDULE                 = "warmup_step_decay"   # "warmup_step_decay" | "cosine" | "linear" | "constant"
DECAY_FRACS              = (0.80, 0.90)
DECAY_GAMMA              = 0.316

# DeepSpeed
ds_config = {
    "train_batch_size": "auto",
    "train_micro_batch_size_per_gpu": "auto",
    "gradient_accumulation_steps": "auto",
    "bf16": {"enabled": True},
    "zero_optimization": {
        "stage": 2,
        "offload_optimizer": {"device": "cpu", "pin_memory": True},
        "overlap_comm": True,
        "contiguous_gradients": True,
        "reduce_bucket_size": 5e8,
        "allgather_bucket_size": 5e8,
    },
    "gradient_clipping": 1.0,
    "steps_per_print": 10,
    "wall_clock_breakdown": False,
    "zero_allow_untested_optimizer": True,
}

ds_config_path = "./deepseekmoe_dynamic_routing_algorithms/ds_config.json"
with open(ds_config_path, "w") as f:
    json.dump(ds_config, f, indent=2)

### Data Preprocessing

Data Preprocessing for Training and Fine Tuning. Run at once for preprocess dataset for training and fine tuning experiments.

In [ ]:
# Preprocess datasets with the parameters
from source.data_preprocessing import run_preprocessing

run_preprocessing(
    max_seq_len=MAX_SEQ_LEN,
    minority_confidence=MINORITY_CONFIDENCE,
    minority_margin_error=MINORITY_MARGIN_ERROR,
    build_slimpajama_data=True,
    build_minipile_data=False,
)

### DeepSeekMoE (Baseline)

In [ ]:
from source.training_utils.training_runner import run_training, BaselineModel, BaselineConfig
from source.data_preprocessing.config import DEFAULT_TRAIN_PATH, DEFAULT_VAL_PATH
from source.training_utils.config import OUTPUT_BASELINE

run_training(
    BaselineModel, BaselineConfig, OUTPUT_BASELINE,
    is_dynmoe=False,
    data_file_path=DEFAULT_TRAIN_PATH,
    val_data_file_path=DEFAULT_VAL_PATH,

    # Model Config (DeepseekConfig)
    n_shared_experts=2,
    n_routed_experts=64,
    num_experts_per_tok=2,
    aux_loss_alpha=0.001,

    # Training Hyperparameters
    MAX_SEQ_LEN=MAX_SEQ_LEN,
    PER_DEVICE_BATCH=PER_DEVICE_BATCH,
    GRAD_ACCUM=GRAD_ACCUM,
    LEARNING_RATE=LEARNING_RATE,
    NUM_EPOCHS=NUM_EPOCHS,
    WARMUP_STEPS=WARMUP_STEPS,
    WEIGHT_DECAY=WEIGHT_DECAY,
    EARLY_STOPPING_PATIENCE=EARLY_STOPPING_PATIENCE,
    EARLY_STOPPING_THRESHOLD=EARLY_STOPPING_THRESHOLD,

    # Optimizer (AdamW)
    ADAM_BETA1=ADAM_BETA1,
    ADAM_BETA2=ADAM_BETA2,
    ADAM_EPS=ADAM_EPS,
    MAX_GRAD_NORM=MAX_GRAD_NORM,

    # LR schedule
    SCHEDULE=SCHEDULE,
    DECAY_FRACS=DECAY_FRACS,
    DECAY_GAMMA=DECAY_GAMMA,

    # DeepSpeed Config
    ds_config="./deepseekmoe_dynamic_routing_algorithms/ds_config.json",
)

### DeepSeekMoE with DYNMoE Routing (Prototype)

In [ ]:
from source.training_utils.training_runner import run_training, DynmoeModel, DynmoeConfig
from source.data_preprocessing.config import DEFAULT_TRAIN_PATH, DEFAULT_VAL_PATH
from source.training_utils.config import OUTPUT_DEEPSEEK_DYNMOE

run_training(
    DynmoeModel, DynmoeConfig, OUTPUT_DEEPSEEK_DYNMOE,
    is_dynmoe=True,
    data_file_path=DEFAULT_TRAIN_PATH,
    val_data_file_path=DEFAULT_VAL_PATH,

    # DYNMoE Model
    n_shared_experts=2,
    n_routed_experts=64,
    aux_loss_alpha=0.008,
    max_active_k=6,
    min_active_k=1,
    threshold_init=-5.85,
    min_routed_experts=1,
    router_bias_update_rate=0.001,

    # Adaptive Tuning Config
    audit_steps=200,
    prune_threshold=0.05,
    min_active_experts=1,
    bias_update_interval=10,
    clear_cache_every=200,

    # Training Hyperparameters
    MAX_SEQ_LEN=MAX_SEQ_LEN,
    PER_DEVICE_BATCH=PER_DEVICE_BATCH,
    GRAD_ACCUM=GRAD_ACCUM,
    LEARNING_RATE=LEARNING_RATE,
    NUM_EPOCHS=NUM_EPOCHS,
    WARMUP_STEPS=WARMUP_STEPS,
    WEIGHT_DECAY=WEIGHT_DECAY,
    EARLY_STOPPING_PATIENCE=EARLY_STOPPING_PATIENCE,
    EARLY_STOPPING_THRESHOLD=EARLY_STOPPING_THRESHOLD,

    # Optimizer (AdamW)
    ADAM_BETA1=ADAM_BETA1,
    ADAM_BETA2=ADAM_BETA2,
    ADAM_EPS=ADAM_EPS,
    MAX_GRAD_NORM=MAX_GRAD_NORM,

    # LR schedule
    SCHEDULE=SCHEDULE,
    DECAY_FRACS=DECAY_FRACS,
    DECAY_GAMMA=DECAY_GAMMA,

    # DeepSpeed Config
    ds_config="./deepseekmoe_dynamic_routing_algorithms/ds_config.json",
)

### Training Comparison: DeepSeekMoE vs DeepSeekMoE with DYNMoE routing

In [ ]:
import matplotlib
matplotlib.use("Agg")          # remove this line if you want inline figures

from source.training_utils.monitoring import (
    plot_training_comparison, print_monitoring_summary,
)
from source.training_utils.config import (
    OUTPUT_BASELINE, OUTPUT_DEEPSEEK_DYNMOE, world_size,
)

TOKENS_PER_STEP = PER_DEVICE_BATCH * GRAD_ACCUM * MAX_SEQ_LEN * world_size

res = plot_training_comparison(
    OUTPUT_BASELINE,                    # baseline run directory
    OUTPUT_DEEPSEEK_DYNMOE,             # DYNMoE run directory
    output_path="training_comparison.png",
    tokens_per_step=TOKENS_PER_STEP,    # 8 * 8 * 384 * 4 = 98,304
    figsize=(15, 16),
    dpi=150,
    annotate_final=True,
    show=False,
)

print_monitoring_summary(res)           # table + DYNMoE - baseline deltas